In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
from pyspark.sql.functions import round, year, month, date_format, col, monotonically_increasing_id, rank
from pyspark.sql import Window

dbutils.widgets.text("folder_path", "/Volumes/workspace/spending/creditspending", "Folder Path")
folder_path = dbutils.widgets.get("folder_path")

schema = StructType([
    StructField('Date', DateType(), True),
    StructField('Vendor', StringType(), True),
    StructField('Credit', DoubleType(), True),
    StructField('Payment', DoubleType(), True),
    StructField('Card#', StringType(), True)
])
df = spark.read.csv(folder_path, header=False, schema=schema)
df = df.withColumn('Credit', round(df['Credit'], 2))
df = df.withColumn('Year', year(df['Date']))
df = df.withColumn('Month', date_format(df['Date'], 'MMM'))
df = df.withColumn('Month#', month(df['Date']))
df = df.withColumn('row_id', monotonically_increasing_id())

# Monthly Spending
monthly_total = df.groupBy('Year', 'Month#', 'Month').sum('Credit').orderBy('Year', 'Month#')
monthly_total = monthly_total.withColumnRenamed('sum(Credit)', 'Total')
monthly_total = monthly_total.withColumn('Total', round(col('Total'), 2))

# Most Expensive Transactions
window_spec = Window.partitionBy('Year', 'Month').orderBy(col('Credit').desc())
most_exp_transactions = df.withColumn('rank', rank().over(window_spec)).filter(col('rank') == 1).select('Year', 'Month', 'Month#', 'Vendor', 'Credit')
most_exp_transactions = most_exp_transactions.orderBy(col('Month#').asc())

# All Monthly Stats
monthly_stats = monthly_total.alias('mt').join(most_exp_transactions, ['Year', 'Month#'], how='left').orderBy('Month#', ascending = True)
monthly_stats = monthly_stats.withColumnRenamed('Total', 'Total Spending').withColumnRenamed('Vendor', 'Most Expensive Vendor').withColumnRenamed('Credit', 'Amount')
monthly_stats = monthly_stats.select('Year', 'mt.Month', 'Total Spending', 'Most Expensive Vendor', 'Amount')
monthly_stats = monthly_stats.withColumnRenamed('Total Spending', 'total_spending').withColumnRenamed('Most Expensive Vendor', 'most_expensive_vendor')

monthly_stats.write.mode("overwrite").saveAsTable("credit_spending")
display(monthly_stats)

Year,Month,total_spending,most_expensive_vendor,Amount
2024,Jul,639.74,"Amazon.ca*RV8W564Y2 AMAZON.CA, ON",320.91
2024,Aug,137.38,"EXXON CHESTNUT MARKET WARWICK, NY 30.46 USD @ 1.383125",42.13
2024,Sep,44.61,"SMVS CANADA NORTH YORK,, ON",23.0
2024,Oct,47.97,"FAMOUS PLAYER 7411QPS BRAMPTON, ON",32.2
2024,Nov,12.42,Spotify P3170603FC Stockholm,12.42
2024,Dec,282.73,"MTO TSD SO ECHANNEL DOWNSVIEW, ON",81.0


Databricks visualization. Run in Databricks to view.